# Lee (2008) — US Senate elections RD

**Paper:** Lee, D.S. (2008). *Randomized Experiments from Non-Random Selection in US House Elections.* J. Econometrics 142(2), 675–697. (bib key `lee2008randomized`)

**Design:** sharp regression discontinuity. **Data:** real `rdrobust::rdrobust_RDsenate` panel (`lee_2008_senate.csv`, n=1390; x = lagged Democratic margin, y = current Democratic vote share).

**What we reproduce:** the incumbency advantage at the cutoff. Conventional local-linear RD at the CCT MSE-optimal bandwidth (Lee Table 1 ≈ 7.99 pp; CCT 2014 Table 4 convention).

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')  # headless-safe (notebooks run under nbclient in CI)
import matplotlib.pyplot as plt
import numpy as np
import statspai as sp
print('statspai', sp.__version__)

In [ ]:
df, _ = sp.replicate('lee_2008')
print(df.shape)
df.head()

In [ ]:
# Conventional local-linear sharp RD, triangular kernel, CCT bandwidth
rd = sp.rdrobust(df, y='y', x='x', c=0,
                 kernel='triangular', bwselect='cct')
conv = rd.diagnostics['conventional']
jump = float(conv['estimate'])
se = float(conv['se'])
h = float(rd.diagnostics['bandwidth_h'])
print(f'Conventional jump: {jump:.3f} pp (SE {se:.3f}) at h={h:.2f}')

In [ ]:
import pandas as pd
tab = pd.DataFrame([
    ['Conventional jump (pp)', jump, 7.99,
     'Lee (2008) Table 1; CCT (2014) Table 4'],
    ['Conventional SE (pp)', se, 1.46, 'StatsPAI vs R rdrobust parity'],
], columns=['quantity', 'StatsPAI', 'Paper', 'source'])
tab

In [ ]:
# Figure: binned RD scatter with the fitted discontinuity
x = df['x'].values; y = df['y'].values
bins = np.linspace(-100, 100, 41)
idx = np.digitize(x, bins)
bx = [x[idx == i].mean() for i in range(1, len(bins)) if (idx == i).any()]
by = [y[idx == i].mean() for i in range(1, len(bins)) if (idx == i).any()]
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(bx, by, s=14, color='#555')
ax.axvline(0, color='k', ls=':')
ax.set_xlabel('Lagged Democratic margin'); ax.set_ylabel('Dem vote share (pp)')
ax.set_title(f'Lee (2008) Senate RD: jump = {jump:.2f} pp')
fig.tight_layout(); fig

In [ ]:
# --- DRIFT GUARD ---
# StatsPAI pins on the real data at the CCT bandwidth (R-parity).
assert abs(jump - 7.414) < 1e-2, jump
assert abs(se - 1.459) < 1e-2, se
assert abs(h - 17.754) < 1e-2, h
# Scientific check: a positive incumbency advantage of several points.
assert 5.0 < jump < 10.0
print(f'OK: Lee (2008) reproduced (jump={jump:.3f} pp at h={h:.2f}).')

**Result.** The conventional local-linear RD recovers an incumbency advantage of ≈ 7.41 pp (SE 1.46) at the CCT MSE-optimal bandwidth (h ≈ 17.75), matching R `rdrobust` to parity and close to Lee's ≈ 7.99. CCT bias-corrected robust inference (the modern standard) is in the modern track of `sp.replicate('lee_2008')`.